# Computed and Experimentally Determined Molecular Properties for Small Organic Molecules Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset URL:**
https://sen.science/doi/10.71728/senscience.8syp-3j8d/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.8syp-3j8d/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Access metadata as a single object

print("\u2500" * 60)
print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Version:", metadata.version)
print("License:", metadata.license)
print("Authors (by @id):")
for author in getattr(metadata, 'author', []):
    print("  -", author['@id'])
print("\u2500" * 60)

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list record sets and their structure using their `@id`.


In [ ]:
# Print overview of record sets with field and column @ids
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No record sets found in this dataset metadata.")
else:
    for rs in record_sets:
        print(f"Record set @id: {rs['@id']}")
        print("Fields:")
        for field in getattr(rs, 'field', []):
            print(f"  - Field @id: {field['@id']}, name: {getattr(field, 'name', 'N/A')}")
        print("Columns:")
        for col in getattr(rs, 'column', []):
            print(f"  - Column @id: {col['@id']}, name: {getattr(col, 'name', 'N/A')}")
        print("\n")
# For demonstration, list all record set @ids for reference.
record_set_ids = [rs['@id'] for rs in record_sets]
if record_set_ids:
    print("Record set @id(s):", record_set_ids)


## 2b. Quick Preview of Record Set Data

If any record sets are present, preview the records using their `@id`.

In [ ]:
# Preview first few records for each record set
if not record_set_ids:
    print("No record set data to preview.")
else:
    for rsid in record_set_ids:
        print(f"--- Records from record set {rsid} ---")
        count = 0
        for x in dataset.records(record_set=rsid):
            pprint.pprint(x)
            count += 1
            if count >= 3:
                break


## 3. Data Extraction

Load data from each record set into a Pandas DataFrame for analysis.

For each record set, we reference via its `@id` as required.

In [ ]:
# Extract data for all record sets
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Record set {rsid} columns:", df.columns.tolist())
        print(df.head())
    else:
        print(f"Record set {rsid} contains no records.")
# For demonstration, select the first available record set
if record_set_ids:
    primary_rs_id = record_set_ids[0]
    print(f"\nPrimary record set for further analysis: {primary_rs_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates filtering, normalization, and grouping using column `@id`.

**Important**: Refer to fields and columns by their `@id` only.

In [ ]:
# Example EDA operations
import numpy as np
from matplotlib import pyplot as plt

# Choose a numeric field (column) @id for demonstration
if record_set_ids:
    primary_df = dataframes[primary_rs_id]
    numeric_columns = []
    # Try to detect numeric columns by dtype
    for col in primary_df.columns:
        if pd.api.types.is_numeric_dtype(primary_df[col]):
            numeric_columns.append(col)
    if numeric_columns:
        numeric_field_id = numeric_columns[0]  # Select the first numeric column @id
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = primary_df[numeric_field_id].mean()
        filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}):")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

        # Try grouping by categorical column @id
        group_columns = [col for col in primary_df.columns if pd.api.types.is_object_dtype(primary_df[col])]
        if group_columns:
            group_field_id = group_columns[0]
            print(f"Grouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped data by {group_field_id} (mean):")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Examples: histogram for a numeric field, scatter plot for numeric/categorical relationship.

In [ ]:
# Simple visualization: histogram and scatter plot
if record_set_ids and numeric_columns:
    primary_df = dataframes[primary_rs_id]
    numeric_field_id = numeric_columns[0]
    plt.figure(figsize=(8, 4))
    plt.hist(primary_df[numeric_field_id].dropna(), bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.show()

    if group_columns:
        group_field_id = group_columns[0]
        plt.figure(figsize=(8, 4))
        # For scatter, use only top N categories
        top_categories = primary_df[group_field_id].value_counts().index[:5].tolist()
        scatter_df = primary_df[primary_df[group_field_id].isin(top_categories)]
        for cat in top_categories:
            cat_df = scatter_df[scatter_df[group_field_id] == cat]
            plt.scatter(cat_df.index, cat_df[numeric_field_id], label=str(cat))
        plt.xlabel("Record Index")
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.legend(title=group_field_id)
        plt.show()
else:
    print("No visualization possible; missing numeric fields.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook used the `mlcroissant` library to load and process FAIR^2 metadata for small organic molecules.
- Dataset structure was explored via `@id` references for record sets, fields, and columns, ensuring reproducibility.
- Data extraction and analysis can be tailored using specific `@id` values, allowing robust data wrangling and visualization.
- The Croissant schema enables seamless metadata integration and provenance tracking across scientific datasets.

_Please refer to the FAIR^2 documentation for detailed data dictionaries and further processing guidance._